In [6]:
from pathlib import Path
import numpy as np
import polars as pl
from tqdm import tqdm
from xgboost import XGBRegressor
import time
from google.colab import drive
import shutil

In [7]:
# Paths
drive.mount("/content/drive")

# Change this if your Google Drive folder name/path is different
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Code/limits-to-arbitrage-machine-learning")

DRIVE_PROCESSED = DRIVE_PROJECT_ROOT / "data" / "processed"
DRIVE_OUTPUT = DRIVE_PROJECT_ROOT / "output"
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

DRIVE_PANEL_PATH = DRIVE_PROCESSED / "gkx_master_panel.parquet"

print("Drive panel path:", DRIVE_PANEL_PATH)
print("Exists on Drive?", DRIVE_PANEL_PATH.exists())

if not DRIVE_PANEL_PATH.exists():
    raise FileNotFoundError(f"Panel not found on Drive: {DRIVE_PANEL_PATH}")

# Copy the parquet from Drive to Colab local disk for faster reading
LOCAL_PROCESSED = Path("/content/data/processed")
LOCAL_PROCESSED.mkdir(parents=True, exist_ok=True)

LOCAL_PANEL_PATH = LOCAL_PROCESSED / "gkx_master_panel.parquet"

if (
    not LOCAL_PANEL_PATH.exists()
    or LOCAL_PANEL_PATH.stat().st_size != DRIVE_PANEL_PATH.stat().st_size
):
    print("Copying panel from Drive to Colab local disk...")
    shutil.copy2(DRIVE_PANEL_PATH, LOCAL_PANEL_PATH)

print("Local panel path:", LOCAL_PANEL_PATH)
print("Exists locally?", LOCAL_PANEL_PATH.exists())
print("Local size GB:", LOCAL_PANEL_PATH.stat().st_size / 1e9)

# Use local parquet for input speed, but save outputs to Drive
PANEL_PATH = LOCAL_PANEL_PATH
OUTPUT = DRIVE_OUTPUT

Mounted at /content/drive
Drive panel path: /content/drive/MyDrive/Thesis_Code/limits-to-arbitrage-machine-learning/data/processed/gkx_master_panel.parquet
Exists on Drive? True
Copying panel from Drive to Colab local disk...
Local panel path: /content/data/processed/gkx_master_panel.parquet
Exists locally? True
Local size GB: 1.226400575


In [23]:
CHAR_COLS = [
    "absacc", "acc", "aeavol", "age", "agr",
    "baspread", "beta", "betasq", "bm", "bm_ia",
    "cash", "cashdebt", "cashpr", "cfp", "cfp_ia",
    "chatoia", "chcsho", "chempia", "chinv", "chmom",
    "chpmia", "chtx", "cinvest", "convind", "currat",
    "depr", "divi", "divo", "dolvol", "dy",
    "ear", "egr", "ep", "gma", "grcapx",
    "grltnoa", "herf", "hire", "idiovol", "ill",
    "indmom", "invest", "lev", "lgr", "maxret",
    "mom12m", "mom1m", "mom36m", "mom6m", "ms",
    "mvel1", "mve_ia", "nincr", "operprof", "orgcap",
    "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale", "pchquick",
    "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
    "pricedelay", "ps", "quick", "rd", "rd_mve",
    "rd_sale", "realestate", "retvol", "roaq", "roavol",
    "roeq", "roic", "rsup", "salecash", "saleinv",
    "salerec", "secured", "securedind", "sgr", "sin",
    "sp", "std_dolvol", "std_turn", "stdacc", "stdcf",
    "tang", "tb", "turn", "zerotrade"
]

FEATURE_COLS = [f"{c}_rank" for c in CHAR_COLS]

MODEL_NAME = "GBRT_94_huber_test"
TARGET_MODE = "next_month"

# Smoke test first:
TEST_YEARS = range(1987, 1992)
# After it works, change to:
# TEST_YEARS = range(1987, 2017)

START_TARGET_MONTH_ID = 1957 * 12 + 3
END_TARGET_MONTH_ID = 2016 * 12 + 12

In [14]:
# Helper Functions
def oos_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    return 1.0 - np.sum((y_true - y_pred) ** 2) / np.sum(y_true ** 2)


def prepare_panel():
    lf = pl.scan_parquet(PANEL_PATH)

    schema = lf.collect_schema()
    missing_chars = [c for c in CHAR_COLS if c not in schema]

    if missing_chars:
        raise ValueError(f"Missing characteristic columns: {missing_chars[:20]}")

    lf = lf.with_columns([
        pl.col("month_id").cast(pl.Int64),
        pl.col("month").dt.year().alias("info_year"),
    ])

    # next-month target
    y_lf = lf.select([
        "permno",
        (pl.col("month_id") - 1).alias("month_id"),
        pl.col("ret_excess").alias("y"),
        pl.col("month").alias("target_month"),
        pl.col("month_id").alias("target_month_id"),
        pl.col("month").dt.year().alias("target_year"),
    ])

    lf = lf.join(y_lf, on=["permno", "month_id"], how="left")

    lf = lf.with_columns(
        pl.col("mktcap").alias("size_for_ranking")
    )

    lf = lf.filter(
        (pl.col("target_month_id") >= START_TARGET_MONTH_ID) &
        (pl.col("target_month_id") <= END_TARGET_MONTH_ID) &
        (pl.col("y").is_not_null())
    )

    # Fill characteristics by monthly median
    lf = lf.with_columns([
        pl.col(c)
        .fill_null(pl.col(c).median().over("month_id"))
        .fill_null(0.0)
        .cast(pl.Float32)
        .alias(f"{c}_filled")
        for c in CHAR_COLS
    ])

    # Rank-transform to [-1, 1]
    lf = lf.with_columns(
        pl.len().over("month_id").alias("_n_month")
    )

    lf = lf.with_columns([
        (
            2.0
            * (pl.col(f"{c}_filled").rank("average").over("month_id") - 1.0)
            / (pl.col("_n_month") - 1.0)
            - 1.0
        )
        .fill_nan(0.0)
        .cast(pl.Float32)
        .alias(f"{c}_rank")
        for c in CHAR_COLS
    ])

    return lf


def collect_xy(lf, feature_cols, train_max_year=None, test_year=None):
    q = lf

    if train_max_year is not None:
        q = q.filter(pl.col("target_year") <= train_max_year)

    if test_year is not None:
        q = q.filter(pl.col("target_year") == test_year)

    id_cols = [
        "permno", "month", "target_month", "month_id",
        "target_month_id", "target_year", "y", "size_for_ranking"
    ]

    df = q.select(id_cols + feature_cols).collect()

    ids = df.select(id_cols)
    X = df.select(feature_cols).to_numpy().astype(np.float32)
    y = df.get_column("y").to_numpy().astype(np.float32)

    return ids, X, y

In [24]:
def run_gbrt():
    lf = prepare_panel()

    all_preds = []

    for test_year in tqdm(TEST_YEARS, desc=MODEL_NAME):
        year_start = time.perf_counter()
        
        train_end_year = test_year - 13

        ids_train, X_train, y_train = collect_xy(
            lf,
            FEATURE_COLS,
            train_max_year=train_end_year
        )

        ids_test, X_test, y_test = collect_xy(
            lf,
            FEATURE_COLS,
            test_year=test_year
        )

        model = XGBRegressor(
            objective="reg:pseudohubererror",
            tree_method="hist",
            max_depth=2,
            n_estimators=100,
            learning_rate=0.03,
            subsample=0.70,
            colsample_bytree=0.70,
            min_child_weight=500,
            reg_lambda=10.0,
            max_bin=32,
            n_jobs=2,
            random_state=42,
            verbosity=1,
        )

        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        out = ids_test.with_columns([
            pl.Series("prediction", pred),
            pl.lit(MODEL_NAME).alias("model"),
            pl.lit(train_end_year).alias("train_end_year"),
        ])

        year_pred_path = OUTPUT / f"predictions_{MODEL_NAME}_{TARGET_MODE}_{test_year}.parquet"
        out.write_parquet(year_pred_path, compression="zstd")
        print(f"Saved yearly predictions: {year_pred_path}")

        all_preds.append(out)

    preds = pl.concat(all_preds)

    pred_path = OUTPUT / f"predictions_{MODEL_NAME}_{TARGET_MODE}.parquet"
    preds.write_parquet(pred_path, compression="zstd")

    r2_all = oos_r2(
        preds.get_column("y").to_numpy(),
        preds.get_column("prediction").to_numpy()
    )

    preds_ranked = (
        preds
        .filter(pl.col("size_for_ranking").is_not_null())
        .with_columns([
            pl.col("size_for_ranking")
            .rank("ordinal", descending=True)
            .over("target_month_id")
            .alias("rank_big"),

            pl.col("size_for_ranking")
            .rank("ordinal", descending=False)
            .over("target_month_id")
            .alias("rank_small"),
        ])
    )

    big = preds_ranked.filter(pl.col("rank_big") <= 1000)
    small = preds_ranked.filter(pl.col("rank_small") <= 1000)

    r2_big = oos_r2(
        big.get_column("y").to_numpy(),
        big.get_column("prediction").to_numpy()
    )

    r2_small = oos_r2(
        small.get_column("y").to_numpy(),
        small.get_column("prediction").to_numpy()
    )

    summary = pl.DataFrame({
        "model": [MODEL_NAME],
        "target_mode": [TARGET_MODE],
        "r2_all_pct": [100 * r2_all],
        "r2_top1000_pct": [100 * r2_big],
        "r2_bottom1000_pct": [100 * r2_small],
        "n_obs": [preds.height],
        "n_features": [len(FEATURE_COLS)],
        "prediction_file": [str(pred_path)],
    })

    print(summary)

    summary.write_csv(OUTPUT / f"r2_summary_{MODEL_NAME}_{TARGET_MODE}.csv")

In [25]:
if __name__ == "__main__":
    run_gbrt()

GBRT_94_huber_test:  20%|██        | 1/5 [05:48<23:14, 348.74s/it]

Saved yearly predictions: /content/drive/MyDrive/Thesis_Code/limits-to-arbitrage-machine-learning/output/predictions_GBRT_94_huber_test_next_month_1987.parquet


GBRT_94_huber_test:  40%|████      | 2/5 [11:35<17:22, 347.52s/it]

Saved yearly predictions: /content/drive/MyDrive/Thesis_Code/limits-to-arbitrage-machine-learning/output/predictions_GBRT_94_huber_test_next_month_1988.parquet


GBRT_94_huber_test:  60%|██████    | 3/5 [17:24<11:36, 348.42s/it]

...

GBRT_94_huber_test:  80%|████████  | 4/5 [23:22<05:52, 352.05s/it]

Saved yearly predictions: /content/drive/MyDrive/Thesis_Code/limits-to-arbitrage-machine-learning/output/predictions_GBRT_94_huber_test_next_month_1990.parquet


GBRT_94_huber_test: 100%|██████████| 5/5 [29:10<00:00, 350.19s/it]

Saved yearly predictions: /content/drive/MyDrive/Thesis_Code/limits-to-arbitrage-machine-learning/output/predictions_GBRT_94_huber_test_next_month_1991.parquet


shape: (1, 8)
┌────────────┬────────────┬────────────┬────────────┬────────────┬────────┬────────────┬───────────┐
│ model      ┆ target_mod ┆ r2_all_pct ┆ r2_top1000 ┆ r2_bottom1 ┆ n_obs  ┆ n_features ┆ predictio │
│ ---        ┆ e          ┆ ---        ┆ _pct       ┆ 000_pct    ┆ ---    ┆ ---        ┆ n_file    │
│ str        ┆ ---        ┆ f64        ┆ ---        ┆ ---        ┆ i64    ┆ i64        ┆ ---       │
│            ┆ str        ┆            ┆ f64        ┆ f64        ┆        ┆            ┆ str       │
╞════════════╪════════════╪════════════╪════════════╪════════════╪════════╪════════════╪═══════════╡
│ GBRT_94_hu ┆ next_month ┆ -3.189799  ┆ -13.838185 ┆ -1.640096  ┆ 408017 ┆ 94         ┆ /content/ │
│ ber_test   ┆            ┆            ┆            ┆            ┆        ┆            ┆ drive/MyD │
│            ┆            ┆            ┆            ┆            ┆        ┆            ┆ rive/Thes │
│            ┆            ┆            ┆            ┆            ┆        ┆  

In [27]:
MODEL_TO_DIAG = "GBRT_94_huber_test"

preds = pl.read_parquet(
    OUTPUT / f"predictions_{MODEL_TO_DIAG}_{TARGET_MODE}.parquet"
)

diag = (
    preds
    .with_columns(pl.col("target_month").dt.year().alias("year"))
    .group_by("year")
    .agg([
        pl.len().alias("n"),
        pl.col("y").mean().alias("y_mean"),
        pl.col("prediction").mean().alias("pred_mean"),
        pl.col("y").std().alias("y_std"),
        pl.col("prediction").std().alias("pred_std"),
        pl.corr("y", "prediction").alias("corr_y_pred"),
        ((pl.col("y") - pl.col("prediction")) ** 2).sum().alias("sse"),
        (pl.col("y") ** 2).sum().alias("sst_zero"),
    ])
    .with_columns(
        (100 * (1 - pl.col("sse") / pl.col("sst_zero"))).alias("r2_pct")
    )
    .sort("year")
)

with pl.Config(tbl_cols=-1, tbl_width_chars=180):
    print(
        diag.select([
            "year",
            "n",
            "y_mean",
            "pred_mean",
            "y_std",
            "pred_std",
            "corr_y_pred",
            "r2_pct",
        ])
    )

shape: (5, 8)
┌──────┬───────┬───────────┬───────────┬──────────┬──────────┬─────────────┬───────────┐
│ year ┆ n     ┆ y_mean    ┆ pred_mean ┆ y_std    ┆ pred_std ┆ corr_y_pred ┆ r2_pct    │
│ ---  ┆ ---   ┆ ---       ┆ ---       ┆ ---      ┆ ---      ┆ ---         ┆ ---       │
│ i32  ┆ u32   ┆ f64       ┆ f32       ┆ f64      ┆ f32      ┆ f64         ┆ f64       │
╞══════╪═══════╪═══════════╪═══════════╪══════════╪══════════╪═════════════╪═══════════╡
│ 1987 ┆ 82108 ┆ -0.008499 ┆ -0.017197 ┆ 0.190205 ┆ 0.013907 ┆ 0.035235    ┆ -0.028728 │
│ 1988 ┆ 84080 ┆ 0.009981  ┆ -0.034479 ┆ 0.163239 ┆ 0.006033 ┆ -0.01018    ┆ -7.228986 │
│ 1989 ┆ 81767 ┆ 0.00294   ┆ -0.02952  ┆ 0.162962 ┆ 0.013545 ┆ 0.055573    ┆ -3.700926 │
│ 1990 ┆ 80600 ┆ -0.025515 ┆ -0.014294 ┆ 0.195277 ┆ 0.017593 ┆ 0.046642    ┆ 1.382206  │
│ 1991 ┆ 79462 ┆ 0.030601  ┆ -0.03578  ┆ 0.230993 ┆ 0.006212 ┆ 0.026804    ┆ -6.320535 │
└──────┴───────┴───────────┴───────────┴──────────┴──────────┴─────────────┴───────────┘


In [21]:
print(
    preds.select([
        pl.col("y").quantile(0.01).alias("y_p01"),
        pl.col("y").quantile(0.50).alias("y_p50"),
        pl.col("y").quantile(0.99).alias("y_p99"),
        pl.col("prediction").quantile(0.01).alias("pred_p01"),
        pl.col("prediction").quantile(0.50).alias("pred_p50"),
        pl.col("prediction").quantile(0.99).alias("pred_p99"),
    ])
)

shape: (1, 6)
┌───────────┬─────────┬────────┬───────────┬───────────┬──────────┐
│ y_p01     ┆ y_p50   ┆ y_p99  ┆ pred_p01  ┆ pred_p50  ┆ pred_p99 │
│ ---       ┆ ---     ┆ ---    ┆ ---       ┆ ---       ┆ ---      │
│ f64       ┆ f64     ┆ f64    ┆ f32       ┆ f32       ┆ f32      │
╞═══════════╪═════════╪════════╪═══════════╪═══════════╪══════════╡
│ -0.432771 ┆ -0.0055 ┆ 0.5954 ┆ -0.060252 ┆ -0.031585 ┆ 0.036607 │
└───────────┴─────────┴────────┴───────────┴───────────┴──────────┘


In [22]:
for test_year in range(1987, 1992):
    train_end_year = test_year - 13

    _, X_train, y_train = collect_xy(
        prepare_panel(),
        FEATURE_COLS,
        train_max_year=train_end_year
    )

    _, X_test, y_test = collect_xy(
        prepare_panel(),
        FEATURE_COLS,
        test_year=test_year
    )

    print(
        test_year,
        "train_end:", train_end_year,
        "train_y_mean:", float(np.mean(y_train)),
        "train_y_std:", float(np.std(y_train)),
        "test_y_mean:", float(np.mean(y_test)),
        "test_y_std:", float(np.std(y_test)),
        "n_train:", len(y_train),
        "n_test:", len(y_test),
    )

1987 train_end: 1974 train_y_mean: -0.00349486549384892 train_y_std: 0.1257656067609787 test_y_mean: -0.008499127812683582 test_y_std: 0.19020362198352814 n_train: 470297 n_test: 82108
1988 train_end: 1975 train_y_mean: 0.0013063240330666304 train_y_std: 0.1342424750328064 test_y_mean: 0.009980782866477966 test_y_std: 0.16323846578598022 n_train: 528528 n_test: 84080
1989 train_end: 1976 train_y_mean: 0.0042769769206643105 train_y_std: 0.1361483484506607 test_y_mean: 0.0029397234320640564 test_y_std: 0.16296085715293884 n_train: 586632 n_test: 81767
1990 train_end: 1977 train_y_mean: 0.005165948532521725 train_y_std: 0.13511478900909424 test_y_mean: -0.025514736771583557 test_y_std: 0.1952761858701706 n_train: 645652 n_test: 80600
1991 train_end: 1978 train_y_mean: 0.006072451826184988 train_y_std: 0.13633865118026733 test_y_mean: 0.030601032078266144 test_y_std: 0.23099130392074585 n_train: 703449 n_test: 79462
